In [94]:
# ----------------------------------------
# 1. Import libraries
# ----------------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [95]:
# ----------------------------------------
# 2. Load the final model-ready dataset
# ----------------------------------------
df = pd.read_csv("../data/final_dataset_model_ready.csv")
df.shape

(612, 12)

In [96]:
# ----------------------------------------
# 3. Define predictor variables and target variable
# ----------------------------------------
FEATURES = [
    "log_gdp",
    "health_expenditure",
    "population_growth",
    "school_enrollment",
    "unemployment",
    "urban_population"
]
TARGET = "shortage"

X = df[FEATURES]
y = df[TARGET]

print("Features shape:", X.shape)
print("\nTarget distribution:")
print(y.value_counts(normalize=True))

Features shape: (612, 6)

Target distribution:
shortage
0    0.697712
1    0.302288
Name: proportion, dtype: float64


In [97]:
# ----------------------------------------
# 4. Split data into training and testing sets (80/20, stratified)
# ----------------------------------------
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape, " | Shortage rate:", round(y_train.mean(), 3))
print("Testing set: ", X_test.shape, " | Shortage rate:", round(y_test.mean(), 3))

Training set: (489, 6)  | Shortage rate: 0.303
Testing set:  (123, 6)  | Shortage rate: 0.301


In [98]:
# ----------------------------------------
# 5. Save the train/test split for reuse in the evaluation notebook
# ----------------------------------------
X_train.to_csv("../outputs/X_train.csv", index=False)
X_test.to_csv("../outputs/X_test.csv", index=False)
y_train.to_csv("../outputs/y_train.csv", index=False)
y_test.to_csv("../outputs/y_test.csv", index=False)

print("Train/test split saved to outputs/")

Train/test split saved to outputs/


In [99]:
# ------------------------------------------
# 6. Baseline Model - Logistic Regression
# ------------------------------------------

from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    class_weight="balanced"
)

log_reg.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [100]:
# ------------------------------------------
# 7. Random Forest Classifier
# ------------------------------------------

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [101]:
# ------------------------------------------
# 8. Support Vector Machine (SVM)
# ------------------------------------------

from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

svm_model.fit(X_train, y_train)

print("SVM trained successfully.")

SVM trained successfully.


In [102]:
# ------------------------------------------
# 9. XGBoost Classifier
# ------------------------------------------

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=3,
    reg_lambda=2.0,
    reg_alpha=0.5,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [103]:
# ------------------------------------------
# 10. Save Trained Models
# ------------------------------------------

import joblib

joblib.dump(log_reg, "../models/logistic_regression.joblib")
joblib.dump(rf_model, "../models/random_forest.joblib")
joblib.dump(svm_model, "../models/svm_rbf.joblib")
joblib.dump(xgb_model, "../models/xgboost.joblib")

print("All models saved to models/")

All models saved to models/


In [104]:
# ------------------------------------------
# 11. Training Accuracy
# ------------------------------------------

models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf_model,
    "SVM (RBF)": svm_model,
    "XGBoost": xgb_model
}

for name, model in models.items():
    train_acc = model.score(X_train, y_train)
    print(f"{name}: training accuracy = {train_acc:.3f}")

Logistic Regression: training accuracy = 0.834
Random Forest: training accuracy = 0.928
SVM (RBF): training accuracy = 0.896
XGBoost: training accuracy = 0.935


In [105]:
# ------------------------------------------
# 7.1 Hyperparameter Tuning - Random Forest (Grid Search)
# ------------------------------------------

from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "min_samples_leaf": [3, 5, 10]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validated F1-Score:", round(grid_search.best_score_, 4))

Best Parameters: {'max_depth': 8, 'min_samples_leaf': 3, 'n_estimators': 100}
Best Cross-Validated F1-Score: 0.8349


In [106]:
# ------------------------------------------
# 9.1 Hyperparameter Tuning - XGBoost (Grid Search)
# ------------------------------------------

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

param_grid = {
    "n_estimators": [100, 150, 300],
    "max_depth": [3, 4, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "reg_lambda": [1.0, 2.0, 5.0]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

grid_search_xgb = GridSearchCV(
    estimator=XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=3,
        reg_alpha=0.5
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1
)

grid_search_xgb.fit(X_train, y_train)

print("Best Parameters:", grid_search_xgb.best_params_)
print("Best Cross-Validated F1-Score:", round(grid_search_xgb.best_score_, 4))

Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 150, 'reg_lambda': 1.0}
Best Cross-Validated F1-Score: 0.8493


In [107]:
# ------------------------------------------
# 7. Random Forest Classifier (Tuned)
# ------------------------------------------

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest (tuned) trained successfully.")

Random Forest (tuned) trained successfully.


In [108]:
# ------------------------------------------
# 9. XGBoost Classifier (Tuned)
# ------------------------------------------

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.1,
    reg_lambda=1.0,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=3,
    reg_alpha=0.5,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

xgb_model.fit(X_train, y_train)

print("XGBoost (tuned) trained successfully.")

XGBoost (tuned) trained successfully.


In [109]:
# ------------------------------------------
# 11. Training Accuracy
# ------------------------------------------

models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf_model,
    "SVM (RBF)": svm_model,
    "XGBoost": xgb_model
}

for name, model in models.items():
    train_acc = model.score(X_train, y_train)
    print(f"{name}: training accuracy = {train_acc:.3f}")

Logistic Regression: training accuracy = 0.834
Random Forest: training accuracy = 0.969
SVM (RBF): training accuracy = 0.896
XGBoost: training accuracy = 0.975


In [110]:
# ------------------------------------------
# 10. Save Trained Models
# ------------------------------------------

import joblib

joblib.dump(log_reg, "../models/logistic_regression.joblib")
joblib.dump(rf_model, "../models/random_forest.joblib")
joblib.dump(svm_model, "../models/svm_rbf.joblib")
joblib.dump(xgb_model, "../models/xgboost.joblib")

print("All models saved to models/")

All models saved to models/
